# Optimizing Agentic AI Collaboration v3.0.0 — L1 Smoke Test: Self-Consistency, Majority Voting & BoN-MAV

**Purpose:** validate the full L1 pipeline end-to-end, before committing GPU-hours to real runs — Self-Consistency's generation/persistence machinery (`utils.persistence`/`utils.voting`/`utils.graphs`) that the 8 real SC notebooks (`v3.1.x`–`v3.3.x`) depend on, **and** Majority Voting (`utils.ensembles`/`build_majority_voting_graph`, both replayed-from-SC and live) **and** BoN-MAV (`utils.mav`/`build_bon_mav_graph`, both replayed-from-SC and live). This is the gate for `v3.1.x`–`v3.3.x` (SC), `v4.x` (Majority Voting), and `v5.x` (BoN-MAV).

**Hardware target:** Nvidia T4 GPU (Kaggle / Colab free tier). Runs against the `dev-agentic-baselines` W&B project, never the real results project.

## Grid

### Self-Consistency Stage

**4 models × 2 benchmarks × 2 problems × 5 generators = 80 generations**, using `SMOKE_DATA_SEED=4242` (different from the real `789`, so this run differs from the real eval subsets) and the full production `GEN` params (unchanged, so parsing/extraction bugs surface). Estimated **1–1.5 h** on a T4 including four model load/unload cycles.

| Model | MATH-500 max_new_tokens | MMLU max_new_tokens |
|---|---|---|
| Ministral-3-3B-Instruct | 1500 | 1024 |
| Qwen3-1.7B | 1500 (`/no_think`) | 1024 (`/no_think`) |
| Qwen3-4B | 1500 (`/no_think`) | 1024 (`/no_think`) |
| Phi-4-mini-instruct | 1500 | 1024 |

### Stages A–C2 (Majority Voting & BoN-MAV)

| Stage | Protocol | GPU | Scope | New live compute |
|---|---|---|---|---|
| A | Majority Voting (replay) | No | `E1` (3 slots) + `E4` (5 slots), both benchmarks, 2 reps each — invariant/replay-fidelity checks; one full MAJ-style W&B run logged for `E1`/MATH-500 | None — replays the 80 SC generations above |
| B | Majority Voting (live) | Yes | `E1` (3 distinct models) + `E4` (4 distinct models) co-resident, MATH-500 only, 2 reps × 2 problems | 32 live generations, estimated 25 mins on Kaggle T4 GPU |
| C | BoN-MAV (SC-reused candidates) | Yes (verifier only) | `qwen3-1.7b` verifier, `E2`'s 3 candidates replayed, MATH-500, 2 problems × 3 aspects | 18 live verifier calls — candidates are reused, not regenerated. Estimated 10 mins on Kaggle T4 GPU |
| C2 | BoN-MAV (live candidates) | Yes (candidates + verifier) | `E2`'s 3 candidates + `qwen3-1.7b` verifier, all co-resident, MATH-500, 2 problems × 3 aspects | 6 live candidate generations + 18 live verifier calls. Estimated 25 mins on Kaggle T4 GPU |


## Relation with the project WPs

**WP2 - T4: L1 multi-agent protocols — Self-Consistency, Majority Voting, BoN-MAV.** This notebook is the validation gate for the whole L1 campaign's generator-compute machinery: the persistence artifact schema and SC@k ladder logic (Self-Consistency), the shared LangGraph voting template in both replay and live modes (Majority Voting, Stages A/B), and the candidate+verifier+selection template in both replay and live modes (BoN-MAV, Stages C/C2) — all exercised on real (small) GPU generations rather than mocked.

---
### Before running (Kaggle)

0. Upload this notebook to a Kaggle kernel (or open it directly if already there).
1. In **Settings**, enable **Accelerator: GPU T4 x2** (the notebook pins itself to one GPU).
2. Add the following secrets via **Add-ons → Secrets**, enabling each for the notebook:
   - `GITHUB_TOKEN` — GitHub PAT with repo read scope
   - `HF_TOKEN` — Hugging Face read-access token
   - `WANDB_API_KEY` — Weights & Biases API key
3. Click **Run All**. The first cell auto-detects Kaggle, clones the repo, and loads secrets.

### Before running (Colab)

0. **Copy this notebook** → File → Save a copy in Drive
1. **Enable GPU** → Runtime → Change runtime type → T4 GPU → Save
2. **Add Colab Secrets** (🔑 icon in the left sidebar):
   - `HF_TOKEN` — your Hugging Face read-access token ([create one here](https://huggingface.co/settings/tokens))
   - `WANDB_API_KEY` — your Weights & Biases API key ([find it here](https://wandb.ai/authorize))
   - `GITHUB_TOKEN` — a GitHub Personal Access Token with `repo` read scope
   - **Enable "Notebook access"** for all secrets

## What this smoke test validates

* The **generation pass** (live LangGraph, one agent, per-seed LLM rebuild) for both MATH-500 and MMLU — including the Qwen3 `/no_think` template variant. Grouping and voting are unified across benchmarks via `utils.voting.MATH500_SPEC`/`MMLU_SPEC` (a `BenchmarkSpec` bound once per adapter), and the vote is produced **by the graph's `aggregate` node itself**.
* **Full-text artifact persistence** (`utils.persistence.log_generations_artifact`) across a resumable per-seed logging loop, and **retrieval** (`load_generations_artifact` via `use_artifact`) in a separate downstream run — proving real W&B producer→consumer lineage, not just in-memory reuse.
* **Alignment validation** (`utils.persistence.validate_alignment`) against the locally loaded smoke subsets, for all 8 producer artifacts.
* The full **SC@1/3/4/5 ladder** (`utils.voting.run_self_consistency` + `compose_generation_metrics`) computed once per (model, benchmark) and logged as 4 separate aggregation runs — the same code path the real notebooks use.
* **Sequential model loading**: exactly one model resident in VRAM at any time, confirmed via `torch.cuda.memory_allocated()` after each unload.
* **Majority Voting replayed from SC** (`utils.ensembles.sample_slots`, `build_majority_voting_graph` in `mode="replay"`) for a 3-slot (`E1`) and a 5-slot, one-duplicated-model (`E4`) ensemble — sampler/seed-reuse/replay-fidelity/determinism/composition invariants, plus one full MAJ-style W&B run (Stage A).
* **Majority Voting live** (`mode="live"`, `E1`/`E4` co-resident) as an equivalence check against the replayed answers, with the VRAM-composition rule validated against the raw co-resident peak (Stage B).
* **BoN-MAV with SC-reused candidates** (`build_bon_mav_graph`, `E2`'s 3 candidates replayed, one live verifier) exercising all 3 `max_new_tokens` strategy tiers (`direct`/`reasoning`/`step_by_step`) (Stage C).
* **BoN-MAV with live candidates** (`mode="live"` for all of `E2`'s candidates, co-resident with the verifier) — proving `build_bon_mav_graph` works unmodified with live candidates, and quantifying the full-live cost against Stage C's reused-vs-live split (Stage C2).

## 1. Setup and Installation

We install the libraries required by the repo. If running on Colab/Kaggle, the repo is cloned first.

In [1]:
# Environment detection: clone private repo
import sys, os

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    ENVIRONMENT = "kaggle"
elif "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ:
    ENVIRONMENT = "colab"
else:
    ENVIRONMENT = "local"


if ENVIRONMENT == "colab":
    repo_root = "/content/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from google.colab import userdata  # type: ignore
                os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()

            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

elif ENVIRONMENT == "kaggle":

    # Kaggle provides 2x T4 GPUs; restrict to a single GPU so that
    # experiments run under the same resource constraints as Colab.
    # This MUST happen before any torch / CUDA import.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    repo_root = "/kaggle/working/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from kaggle_secrets import UserSecretsClient  # type: ignore
                _secrets = UserSecretsClient()
                os.environ.setdefault("GITHUB_TOKEN", _secrets.get_secret("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()
            from kaggle_secrets import UserSecretsClient  # type: ignore
            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"repo_root = {repo_root}")

ℹ️  Repo already present at /content/tfg-agentic-ai
repo_root = /content/tfg-agentic-ai


In [2]:
# Only necessary if running in Google Colab, but won't cause issues if run locally
%cd {repo_root}
%pip install -r requirements.txt

/content/tfg-agentic-ai


## 2. Authentication via Colab / Kaggle Secrets

We pull `HF_TOKEN` and `WANDB_API_KEY` from Colab userdata or Kaggle Secrets — no tokens in code.

In [3]:
from utils.config import load_env

print(f"Runtime environment: {ENVIRONMENT}")

load_env(environment=ENVIRONMENT)

Runtime environment: colab
✅ Secrets loaded from Colab userdata.


## 3. Smoke Configuration

The 4 target models are looked up from `utils.registry.MODELS` — the single source of truth for `model_id`/`run_name`/`no_think` shared with every production notebook, not a separate list duplicated here. `SMOKE_DATA_SEED=4242` and `SMOKE_EVAL_SIZE=2` keep this run tiny and disjoint from the real `data_seed=789` subsets. `SC_SEEDS`/`TIE_BREAK_SEED`/`K_LADDER` match production exactly — this is what makes the smoke test a real validation of the production code path, not a simplified stand-in.

In [4]:
from utils.registry import MODELS

PROJECT = "l1-smoke-tests"
print(f"W&B Project: {PROJECT}")

MODEL_KEYS = list(MODELS)

# Same sampling params as production (T=0.7, top_p=0.95, top_k=40 per Wang et al.,
# Self-Consistency)
GEN_COMMON = {"temperature": 0.7, "top_p": 0.95, "top_k": 40}

# Different from the real data_seed=789 -> this run never touches the
# real eval subsets. eval_size=2 keeps 4 models x 2 benchmarks x 5 seeds tractable.
SMOKE_DATA_SEED = 4242
SMOKE_EVAL_SIZE = 2

SC_SEEDS       = [42, 123, 456, 654, 987]
TIE_BREAK_SEED = 2026
K_LADDER       = [1, 3, 4, 5]

print(f"✅ {len(MODEL_KEYS)} models x 2 benchmarks x {SMOKE_EVAL_SIZE} problems x {len(SC_SEEDS)} generators "
      f"= {len(MODEL_KEYS) * 2 * SMOKE_EVAL_SIZE * len(SC_SEEDS)} generations planned.")

W&B Project: l1-smoke-tests
✅ 4 models x 2 benchmarks x 2 problems x 5 generators = 80 generations planned.


## 4. Data Loading

Both datasets are loaded once and shared across all 4 models. The 2-problem smoke subsets are drawn with `SMOKE_DATA_SEED=4242` — the same subsets every model in this notebook evaluates against.

In [5]:
import random
from utils.data import load_math500, load_mmlu

math500_dataset = load_math500(split="test")
mmlu_dataset    = load_mmlu(split="test")

random.seed(SMOKE_DATA_SEED)
math_smoke_sample = random.sample(list(math500_dataset), SMOKE_EVAL_SIZE)
random.seed(SMOKE_DATA_SEED)
mmlu_smoke_sample = random.sample(list(mmlu_dataset), SMOKE_EVAL_SIZE)

print(f"✅ MATH-500: {len(math500_dataset)} loaded, {len(math_smoke_sample)} sampled for smoke.")
for i, item in enumerate(math_smoke_sample):
    print(f"  [{i}] {item['problem'][:70]!r}")

print(f"\n✅ MMLU: {len(mmlu_dataset)} loaded, {len(mmlu_smoke_sample)} sampled for smoke.")
for i, item in enumerate(mmlu_smoke_sample):
    print(f"  [{i}] {item['question'][:70]!r}")

✅ MATH-500: 366 loaded, 2 sampled for smoke.
  [0] "What's the largest eight-digit base 2 integer?  Express your answer in"
  [1] 'Find the largest value of $x$ that satisfies the equation $|5x-1|=x+3$'

✅ MMLU: 14042 loaded, 2 sampled for smoke.
  [0] "We're perfectly entitled to believe there is a God. After all, every e"
  [1] 'Find the value of 6 • 24 ÷ (8 − 5).'


## 5. Templates and the Instrumented Generation Closure

`build_templates` produces the MATH-500 and MMLU chain templates for a given model — identical wording to the production notebooks, with the `/no_think` prefix added only for Qwen3 models. `make_generate_fn` is the same instrumented closure builder every production SC notebook uses — and the one Stages B and C2 below reuse unmodified for their own live Majority Voting/BoN-MAV generation, not a separate implementation.

In [6]:
import time
import wandb
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from utils.models import load_model_and_tokenizer
from utils.instrumentation import timed_invoke, tokens_per_second


def build_templates(no_think: bool) -> dict:
    """MATH-500 and MMLU chain templates, identical wording to production —
    Qwen3 models additionally get the /no_think prefix (thinking mode off)."""

    prefix = "/no_think\n\n" if no_think else ""
    math_template = PromptTemplate.from_template(
        prefix + "Problem:\n{task}\n\n"
        "Please reason step by step, and put your final answer within \\boxed{{}}."
    )
    mmlu_template = PromptTemplate.from_template(
        prefix + "Your answer should be of the following format: 'ANSWER: [LETTER]' "
        "(without quotes) where [LETTER] is one of A,B,C,D. "
        "Think step by step before answering.\n\n{task}"
    )

    return {"math500": math_template, "mmlu": mmlu_template}


def make_generate_fn(chain, prompt_template, model_key, model_id, agent_id, gen_id, extract_fn, tokenizer):
    """Build a live ``generate(task, seed) -> AgentSample`` closure — identical
    to the one every production SC notebook defines, via ``timed_invoke``.
    ``tokenizer`` is passed in explicitly (rather than captured from a
    module-level global) because this notebook loads a different tokenizer per
    model in a loop."""

    def generate(task, seed):
        r = timed_invoke(chain, prompt_template, tokenizer, {"task": task})
        return {
            "agent_id": agent_id, "model_key": model_key, "model_id": model_id, "source": "live",
            "gen_id": gen_id, "seed": seed,
            "answer": r["output_text"], "parsed_output": extract_fn(r["output_text"]),
            "latency_s": r["latency_s"], "input_tokens": r["input_tokens"],
            "output_tokens": r["output_tokens"], "total_tokens": r["total_tokens"],
            "gpu_peak_mb": r["gpu_peak_mb"],
        }
    return generate

print("✅ Template builder and generation closure builder ready.")

✅ Template builder and generation closure builder ready.


## 6. Dataset Adapters

To drive one shared pipeline over both benchmarks without duplicating it 4×2 times (as the production notebooks intentionally do, for per-notebook independence), this smoke test factors the benchmark-specific pieces into two small adapter dicts. Extraction/equivalence/scoring are **not** duplicated here — each adapter just points at the shared `utils.voting.MATH500_SPEC`/`MMLU_SPEC`, exactly like the production notebooks' `SPEC`. The adapter only carries what's genuinely per-benchmark and not already owned by `BenchmarkSpec`: `max_new_tokens` and the W&B metric prefix. Task rendering and problem identity are not adapter concerns — `spec.render_task` builds the task text and the loop's own ordinal `problem_id` is the identity, exactly like the production notebooks. Everything downstream (`utils.graphs`, `utils.voting`, `utils.persistence`) is the exact same production code.

In [7]:
from utils.voting import MATH500_SPEC, MMLU_SPEC

# Adapters carry only max_new_tokens and metric_prefix (see §6 above for why).
MATH_ADAPTER = {
    "spec": MATH500_SPEC,
    "max_new_tokens": 1500,
    "metric_prefix": "math_sample",
}

MMLU_ADAPTER = {
    "spec": MMLU_SPEC,
    "max_new_tokens": 1024,
    "metric_prefix": "mmlu_sample",
}

print("✅ MATH-500 and MMLU adapters ready.")

✅ MATH-500 and MMLU adapters ready.


## 7. Pipeline Functions

Two functions, each mirroring one section of the production notebooks exactly:

* `run_generation_pass` — the live LangGraph generation loop (5 seeds × `SMOKE_EVAL_SIZE` problems), per-seed artifact re-logging, generator table, `finish_wandb()`. Returns the in-memory records and the final producer artifact ref.
* `run_sc_k_aggregation` — one W&B run per `k ∈ {1,3,4,5}`, `use_artifact` lineage, reads each k's subset results off `utils.voting.run_self_consistency`'s output (called once in the main loop, per model×benchmark) + `compose_generation_metrics`, `accuracy_statistics`.

Grouping the 5 generations per problem happens inside `run_self_consistency`, not as a separate step here — it groups once per problem, internally, regardless of how many `k` values are requested.

In [8]:
import warnings
warnings.filterwarnings("ignore")

import math
import numpy as np
from utils.agents import build_hf_langchain_llm
from utils.graphs import AgentSpec, build_majority_voting_graph
from utils.registry import generations_artifact_name
from utils.persistence import (
    build_generation_store, make_generation_record, artifact_ref,
    log_generations_artifact, load_generations_artifact,
    validate_alignment, AlignmentError, fetch_and_validate_stores,
)
from utils.voting import compose_generation_metrics, run_self_consistency
from utils.statistics import accuracy_statistics, time_statistics, token_statistics, vram_statistics
from utils.logging_utils import init_wandb, define_wandb_chart_only_metrics, log_metrics, log_stat_summary, finish_wandb


def run_generation_pass(model, tokenizer, model_key, model_cfg, dataset_key, dataset_sample, template, adapter):
    """Live generation pass for one (model, benchmark): 5 seeds x len(dataset_sample)
    problems, through a single-live-agent LangGraph, with per-seed resumable
    artifact logging — identical structure to the production notebooks' cell 10."""

    model_id  = model_cfg["model_id"]
    run_name  = model_cfg["run_name"]
    spec = adapter["spec"]
    max_new_tokens = adapter["max_new_tokens"]
    metric_prefix  = f"{adapter["metric_prefix"]}_generation"
    gen_config = {"max_new_tokens": max_new_tokens, **GEN_COMMON}

    run_config = {
        "model_key": model_key, "model_id": model_id, "quantization": "nf4-4bit",
        "framework": "langchain+langgraph", "protocol": "generation",
        "eval_size": len(dataset_sample), "data_seed": SMOKE_DATA_SEED,
        "generation_seeds": SC_SEEDS, "tie_break_seed": TIE_BREAK_SEED, "k_ladder": K_LADDER,
        **gen_config, "eval_dataset": dataset_key,
    }
    run = init_wandb(
        run_name=f"l1-sc-gen-{run_name}-{dataset_key}-smoke", project=PROJECT, config=run_config,
        tags=["generation", dataset_key, "smoke"],
    )
    define_wandb_chart_only_metrics(prefixes=[f"{metric_prefix}/*"])

    # A single agent never ties (nothing to vote against), so this RNG never
    # actually draws — it exists only to satisfy build_majority_voting_graph's
    # signature, exactly like the production notebooks' gen_tie_rng.
    gen_tie_rng = random.Random(TIE_BREAK_SEED)

    artifact_name = generations_artifact_name(model_key, dataset_key) + "-smoke"
    generation_records = []
    producer_ref = None

    for gen_id, seed in enumerate(SC_SEEDS):
        llm = build_hf_langchain_llm(
            model=model, tokenizer=tokenizer, max_new_tokens=max_new_tokens,
            temperature=gen_config["temperature"], top_p=gen_config["top_p"], top_k=gen_config["top_k"],
            return_full_text=False, seed=seed,
        )
        chain = template | llm | StrOutputParser()
        generate_fn = make_generate_fn(chain, template, model_key, model_id, "gen", gen_id, spec.extract, tokenizer)
        agent_spec = AgentSpec(agent_id="gen", model_key=model_key, model_id=model_id, mode="live", generate=generate_fn, seed=seed)
        graph = build_majority_voting_graph([agent_spec], spec, rng=gen_tie_rng)

        for problem_id, item in enumerate(dataset_sample):
            task  = spec.render_task(item)
            truth = spec.get_truth(item)

            out = graph.invoke({
                "task": task, "problem_id": problem_id,
                "dataset": dataset_key, "samples": [], "vote": None, "metrics": None,
            })
            sample = out["samples"][0]
            compare_value = sample["answer"] if spec.compare_on == "answer" else sample["parsed_output"]
            correct = spec.score(compare_value, truth)
            tps     = tokens_per_second(sample["output_tokens"], sample["latency_s"])

            print(f"  ⏱  {model_key}/{dataset_key} gen_id={gen_id} problem={problem_id}: "
                  f"{sample['latency_s']:.1f}s  correct={correct}")

            generation_records.append(make_generation_record(
                dataset=dataset_key, problem_id=problem_id, model_key=model_key, gen_id=gen_id,
                seed=seed, protocol="generation", task=task, sample=sample,
                correct=correct, tokens_per_s=tps,
            ))
            log_metrics({
                f"{metric_prefix}/accuracy": int(correct),
                f"{metric_prefix}/latency_s": sample["latency_s"],
                f"{metric_prefix}/output_tokens": sample["output_tokens"],
                f"{metric_prefix}/total_tokens": sample["total_tokens"],
                f"{metric_prefix}/tokens_per_s": tps,
                f"{metric_prefix}/gpu_peak_mb": sample["gpu_peak_mb"],
                })

        manifest = {
            "model_key": model_key, "dataset": dataset_key,
            "protocol": "self-consistency", "data_seed": SMOKE_DATA_SEED, "eval_size": len(dataset_sample),
            "generation_seeds": SC_SEEDS[:gen_id + 1], "gen_params": gen_config,
            "wandb_run_id": run.id, "wandb_run_name": run.name, "wandb_project": PROJECT,
            "n_records": len(generation_records),
            "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }
        artifact = log_generations_artifact(generation_records, manifest, name=artifact_name, run=run)
        # Captured off the actually-committed version (artifact.wait()'d inside
        # log_generations_artifact) — the round-trip validation cell downloads
        # exactly this pinned ref, not a computed ":vN" guess.
        producer_ref = artifact_ref(artifact, entity=run.entity, project=PROJECT)
        wandb.summary["producer_artifact"] = producer_ref
        wandb.summary["producer_artifact_version"] = artifact.version

    finish_wandb()

    return generation_records, producer_ref


def run_sc_k_aggregation(model_key, model_cfg, dataset_key, producer_ref, store, sc_all, dataset_sample, adapter):
    """One W&B run per k in K_LADDER — each run downloads its own copy of the
    generation artifact via load_generations_artifact(run=sc_run), so every
    aggregation run records real producer->consumer lineage on itself (not just
    a bare use_artifact call) — reads each problem's subset results off sc_all
    (utils.voting.run_self_consistency's output, computed once in the main loop)
    + compose_generation_metrics, accuracy_statistics — the same functions the
    production SC@k cell calls."""

    model_id = model_cfg["model_id"]
    run_name = model_cfg["run_name"]

    n_generators = len(SC_SEEDS)
    sample_size  = len(dataset_sample)

    spec = adapter["spec"]
    metric_prefix = adapter["metric_prefix"]

    k_results = {}
    for k in K_LADDER:
        n_subsets = math.comb(n_generators, k)

        sc_run = init_wandb(
            run_name=f"l1-sc{k}-{run_name}-{dataset_key}-smoke", project=PROJECT,
            config={
                "model_key": model_key, "model_id": model_id, "protocol": "self-consistency",
                "eval_size": sample_size, "data_seed": SMOKE_DATA_SEED, "generation_seeds": SC_SEEDS,
                "tie_break_seed": TIE_BREAK_SEED, "sc_k": k, "n_subsets": n_subsets,
                "producer_artifact": producer_ref, "eval_dataset": dataset_key,
            },
            tags=["l1", "self-consistency", f"sc{k}", dataset_key, "smoke"],
        )
        records, manifest = load_generations_artifact(producer_ref, run=sc_run)
        store = build_generation_store(records)
        define_wandb_chart_only_metrics(prefixes=[f"{metric_prefix}/*"])

        correctness_matrix = np.zeros((sample_size, n_subsets), dtype=int)
        time_matrix        = np.zeros((sample_size, n_subsets), dtype=float)
        out_tok_matrix    = np.zeros((sample_size, n_subsets), dtype=int)
        total_tok_matrix   = np.zeros((sample_size, n_subsets), dtype=int)
        gpu_peak_matrix     = np.zeros((sample_size, n_subsets), dtype=float)
        tps_matrix          = np.zeros((sample_size, n_subsets), dtype=float)

        results_columns = [
          "dataset", "model_key", "protocol", "problem_id", "subset_id",
          "winner_gen_id", "task", "ground_truth", "final_output", "correct",
          "pct_votes_top", "tie", "n_abstain",
          "latency_s", "output_tokens", "total_tokens", "gpu_peak_mb", "throughput_tok_s",
        ]
        n_tie = 0
        n_abstain_total = 0
        results_rows = []

        for problem_id in range(sample_size):
            subset_results = sc_all[problem_id]["by_k"][k]
            records = [store[(problem_id, g)] for g in range(n_generators)]
            ground_truth = spec.get_truth(dataset_sample[problem_id])
            task = spec.render_task(dataset_sample[problem_id])

            for subset_idx, sr in enumerate(subset_results):
                metrics = compose_generation_metrics([records[i] for i in sr["indices"]])
                correctness_matrix[problem_id, subset_idx] = int(sr["correct"])
                time_matrix[problem_id, subset_idx]        = metrics["latency_s"]
                out_tok_matrix[problem_id, subset_idx]   = metrics["output_tokens"]
                total_tok_matrix[problem_id, subset_idx]  = metrics["total_tokens"]
                gpu_peak_matrix[problem_id, subset_idx]     = metrics["gpu_peak_mb"]
                tps_matrix[problem_id, subset_idx]       = metrics["tokens_per_s"]
                if sr["tie"]:
                    n_tie += 1
                n_abstain_total += sr["n_abstain"]

                results_rows.append([
                    dataset_key, model_key, f"SC@{k}",
                    problem_id, subset_idx, sr["winner_gen_id"],
                    task, ground_truth, sr["winner_output"],
                    int(sr["correct"]), sr["pct_votes_top"], sr["tie"],
                    sr["n_abstain"], metrics["latency_s"],
                    metrics["output_tokens"], metrics["total_tokens"],
                    metrics["gpu_peak_mb"], metrics["tokens_per_s"],
                ])

            # One chart point per problem — the mean across that problem's
            # subsets, not one point per subset.
            log_metrics({
                f"{metric_prefix}/accuracy":    float(correctness_matrix[problem_id].mean()),
                f"{metric_prefix}/latency_s":   float(time_matrix[problem_id].mean()),
                f"{metric_prefix}/output_tokens": float(out_tok_matrix[problem_id].mean()),
                f"{metric_prefix}/total_tokens": float(total_tok_matrix[problem_id].mean()),
                f"{metric_prefix}/gpu_peak_mb": float(gpu_peak_matrix[problem_id].max()),
                f"{metric_prefix}/throughput_tps": float(tps_matrix[problem_id].mean()),
            })

        acc_stats = accuracy_statistics(correctness_matrix)
        time_stats = time_statistics(time_matrix)
        out_tok_stats = token_statistics(out_tok_matrix)
        total_tok_stats = token_statistics(total_tok_matrix)
        vram_stats = vram_statistics(gpu_peak_matrix)

        wandb.summary["accuracy"]     = round(acc_stats["p_hat"], 6)
        wandb.summary["n_subsets"]    = n_subsets
        wandb.summary["tie_rate"]     = round(n_tie / (sample_size * n_subsets), 4)
        wandb.summary["abstain_rate"] = round(n_abstain_total / (sample_size * n_subsets * n_generators), 4)
        log_stat_summary("accuracy_stats", acc_stats, keys=["ci_lower", "ci_upper"])
        wandb.summary["latency"] = round(time_stats["mean"], 3)
        wandb.summary["output_tokens"] = round(out_tok_stats["mean"], 3)
        wandb.summary["total_tokens"] = round(total_tok_stats["mean"], 3)
        wandb.summary["gpu_peak"] = round(vram_stats["peak_max"], 3)

        results_table = wandb.Table(columns=results_columns, data=results_rows)
        wandb.log({f"results_comparision_MATH500": results_table})

        finish_wandb()

        k_results[k] = {
            "accuracy": acc_stats["p_hat"], "n_subsets": n_subsets,
            "correctness_matrix": correctness_matrix, "gpu_peak_matrix": gpu_peak_matrix,
        }
        print(f"  ✅ {model_key}/{dataset_key} SC@{k}: accuracy={acc_stats['p_hat']:.3f} "
              f"(n_subsets={n_subsets})")

    return k_results

print("✅ Pipeline functions (run_generation_pass, run_sc_k_aggregation) ready.")

✅ Pipeline functions (run_generation_pass, run_sc_k_aggregation) ready.


## 8. Main Loop — 4 Models × 2 Benchmarks
For each model: load → warmup → for each benchmark: run the generation pass, then `utils.voting.run_self_consistency` (one fresh `random.Random(TIE_BREAK_SEED)` per benchmark, mirroring the production notebooks' Vote Once cell) → the SC@k aggregation runs → unload and assert VRAM is released before the next model. Exactly one model is ever resident, matching the production single-model-per-notebook VRAM budget.

In [9]:
import gc
import torch
from utils.warmup import warmup_model

producer_refs      = {}   # (model_key, dataset_key) -> artifact ref
in_memory_records  = {}   # (model_key, dataset_key) -> generation_records
sc_results         = {}   # (model_key, dataset_key) -> {k: {...}}

DATASETS = [
    ("math500", math_smoke_sample, MATH_ADAPTER),
    ("mmlu", mmlu_smoke_sample, MMLU_ADAPTER),
]

for model_key in MODEL_KEYS:
    model_cfg = MODELS[model_key]
    print(f"\n{'#'*70}\n  MODEL: {model_cfg['model_id']}\n{'#'*70}")

    model, tokenizer = load_model_and_tokenizer(model_cfg["model_id"], quantize_4bit=True)
    print(f"✅ Model loaded on {model.device} — dtype {model.dtype}")

    warmup_model(model, tokenizer, max_new_tokens=64, n_iterations=1,
                 temperature=GEN_COMMON["temperature"], top_p=GEN_COMMON["top_p"], top_k=GEN_COMMON["top_k"])

    templates = build_templates(no_think=model_cfg["no_think"])

    for dataset_key, dataset_sample, adapter in DATASETS:
        print(f"\n{'-'*70}\n  {model_key} / {dataset_key}\n{'-'*70}")

        records, producer_ref = run_generation_pass(
            model, tokenizer, model_key, model_cfg, dataset_key, dataset_sample, templates[dataset_key], adapter,
        )
        key = (model_key, dataset_key)
        in_memory_records[key] = records
        producer_refs[key] = producer_ref

        # Fresh, independently seeded RNG — reproducible on its own, mirroring the
        # production notebooks' Vote Once cell.
        store = build_generation_store(records)
        vote_tie_rng = random.Random(TIE_BREAK_SEED)
        sc_all = run_self_consistency(
            store, dataset_sample, adapter["spec"], K_LADDER,
            n_generators=len(SC_SEEDS), rng=vote_tie_rng,
            model_key=model_key, model_id=model_cfg["model_id"],
        )
        k_results = run_sc_k_aggregation(model_key, model_cfg, dataset_key, producer_ref, store, sc_all, dataset_sample, adapter)
        sc_results[key] = k_results

    # Unload before the next model — assert VRAM actually drops
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    allocated_mb = torch.cuda.memory_allocated() / 1e6
    print(f"\nVRAM after unloading {model_key}: {allocated_mb:.1f} MB")
    assert allocated_mb < 1000, (
        f"VRAM not released after unloading {model_cfg['model_id']}: {allocated_mb:.1f} MB"
    )

print(f"\n{'='*70}\n  MAIN LOOP COMPLETE — {len(in_memory_records)} (model, dataset) pairs generated.\n{'='*70}")


######################################################################
  MODEL: Qwen/Qwen3-1.7B
######################################################################


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'top_k', 'max_new_tokens', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Model loaded on cuda:0 — dtype torch.float16
🔥 Warming up model (1 iterations, max_new_tokens=64)…


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


...

mmlu_sample/accuracy,▁█
mmlu_sample/gpu_peak_mb,█▁
mmlu_sample/latency_s,▁█
mmlu_sample/output_tokens,▁█
mmlu_sample/throughput_tps,█▁
mmlu_sample/total_tokens,▁█
abstain_rate,0
accuracy,0.5
accuracy_stats/ci_lower,0
accuracy_stats/ci_upper,1
gpu_peak,2974.596


  ✅ phi-4-mini/mmlu SC@4: accuracy=0.500 (n_subsets=5)


wandb:   2 of 2 files downloaded.  


mmlu_sample/accuracy,▁█
mmlu_sample/gpu_peak_mb,█▁
mmlu_sample/latency_s,▁█
mmlu_sample/output_tokens,▁█
mmlu_sample/throughput_tps,█▁
mmlu_sample/total_tokens,▁█
abstain_rate,0
accuracy,0.5
accuracy_stats/ci_lower,0
accuracy_stats/ci_upper,1
gpu_peak,2974.596


  ✅ phi-4-mini/mmlu SC@5: accuracy=0.500 (n_subsets=1)

VRAM after unloading phi-4-mini: 8.5 MB

  MAIN LOOP COMPLETE — 8 (model, dataset) pairs generated.


## 9. Reuse Round-Trip Validation

With **no model loaded**, download all 8 producer artifacts via `use_artifact` (real W&B lineage, explicit `:v0`-through-final version refs already captured in `producer_refs`), rebuild each store, and validate alignment against the locally loaded smoke subsets — exactly the workflow Majority Voting and BoN-MAV notebooks will use against the real artifacts.

In [10]:
print(f"\n{'='*70}\n  REUSE ROUND-TRIP VALIDATION (no model loaded)\n{'='*70}")

round_trip_run = init_wandb(
    run_name="l1-sc-reuse-roundtrip-smoke", project=PROJECT,
    config={"purpose": "round-trip validation", "producer_refs": {f"{k[0]}/{k[1]}": v for k, v in producer_refs.items()}},
    tags=["l1", "self-consistency", "smoke", "roundtrip"],
)

downloaded_stores = {}
downloaded_manifests = {}
alignment_errors = []

for model_key in MODEL_KEYS:
    for dataset_key, dataset_sample, adapter in DATASETS:
        key = (model_key, dataset_key)
        ref = producer_refs[key]

        records, manifest = load_generations_artifact(ref, run=round_trip_run)
        store = build_generation_store(records)
        downloaded_stores[key] = store
        downloaded_manifests[key] = manifest

        try:
            validate_alignment(
                manifest, store, dataset_sample, render_task=adapter["spec"].render_task,
                expected_dataset=dataset_key, expected_model_key=model_key,
                expected_data_seed=SMOKE_DATA_SEED, expected_eval_size=SMOKE_EVAL_SIZE,
                n_generators=len(SC_SEEDS),
            )
            print(f"  ✅ alignment OK: {key[0]}/{key[1]}  ({ref})")
        except AlignmentError as e:
            alignment_errors.append((key, str(e)))
            print(f"  ❌ alignment FAILED: {key[0]}/{key[1]}: {e}")

assert not alignment_errors, f"Alignment failures: {alignment_errors}"

# Byte-compare fidelity: a downloaded record's full text vs the in-memory original.
fidelity_key = (MODEL_KEYS[0], "math500")
fidelity_store = downloaded_stores[fidelity_key]
fidelity_original = in_memory_records[fidelity_key][0]
fidelity_downloaded = fidelity_store[(fidelity_original["problem_id"], fidelity_original["gen_id"])]
assert fidelity_downloaded["answer"] == fidelity_original["answer"], (
    "Artifact round-trip fidelity FAILED: `answer` text mismatch after download."
)
print(f"\n✅ Artifact round-trip fidelity confirmed: downloaded `answer` is byte-identical "
      f"to the in-memory original ({len(fidelity_original['answer'])} chars).")

finish_wandb()


  REUSE ROUND-TRIP VALIDATION (no model loaded)


wandb:   2 of 2 files downloaded.  


  ✅ alignment OK: qwen3-1.7b/math500  (tfg-optimizing-agentic-ai/l1-smoke-tests/sc-generations-qwen3-1.7b-math500-smoke:v34)


wandb:   2 of 2 files downloaded.  


  ✅ alignment OK: qwen3-1.7b/mmlu  (tfg-optimizing-agentic-ai/l1-smoke-tests/sc-generations-qwen3-1.7b-mmlu-smoke:v9)


...

  ✅ alignment OK: ministral-3-3b/mmlu  (tfg-optimizing-agentic-ai/l1-smoke-tests/sc-generations-ministral-3-3b-mmlu-smoke:v4)


wandb:   2 of 2 files downloaded.  


  ✅ alignment OK: phi-4-mini/math500  (tfg-optimizing-agentic-ai/l1-smoke-tests/sc-generations-phi-4-mini-math500-smoke:v4)


wandb:   2 of 2 files downloaded.  


  ✅ alignment OK: phi-4-mini/mmlu  (tfg-optimizing-agentic-ai/l1-smoke-tests/sc-generations-phi-4-mini-mmlu-smoke:v4)

✅ Artifact round-trip fidelity confirmed: downloaded `answer` is byte-identical to the in-memory original (1095 chars).


## 10. Smoke Stage A — Majority Voting from the SC Stores (replay, no GPU)

Validates that Majority Voting is built entirely from `downloaded_stores` (the round-trip stage's own downloaded stores, not the in-memory records) — proving MAJ works against real downloaded artifacts, the same way the production `v4.1.0` notebook will. Runs on both benchmarks for `E1` (3 slots, `qwen3-1.7b`+`ministral-3-3b`+`phi-4-mini`) and `E4` (5 slots, all four models plus a duplicated `qwen3-1.7b`), 2 reps each. No GPU — every candidate is replayed from the smoke generations already produced by the main loop.

In [11]:
print(f"\n{'='*70}\n  SMOKE STAGE A — Majority Voting from the SC Stores (replay, no GPU)\n{'='*70}")

from utils.ensembles import E1, E4, SLOT_SAMPLE_SEED, sample_slots, slot_agent_ids
from utils.graphs import AgentSpec, build_majority_voting_graph
from utils.voting import compose_generation_metrics

STAGE_A_ENSEMBLES = (E1, E4)
STAGE_A_DATASETS = [
    ("math500", math_smoke_sample, MATH_ADAPTER["spec"]),
    ("mmlu", mmlu_smoke_sample, MMLU_ADAPTER["spec"]),
]
N_REPS = 2

print(f"\n{'-'*70}\n  Sampler + seed-reuse invariants\n{'-'*70}")

draws_by_key = {}  # (ensemble_id, dataset_key, rep, problem_id) -> draw

for ens in STAGE_A_ENSEMBLES:
    for dataset_key, dataset_sample, spec in STAGE_A_DATASETS:
        for rep in range(N_REPS):
            for problem_id in range(len(dataset_sample)):
                draw = sample_slots(ens, dataset=dataset_key, problem_id=problem_id, rep=rep)
                draws_by_key[(ens.ensemble_id, dataset_key, rep, problem_id)] = draw
                for model_key, gen_id in draw:
                    assert 0 <= gen_id < 5, f"gen_id out of range: {draw}"

        # A duplicated model_key (E4's 2nd qwen3-1.7b) must draw two DISTINCT gen_ids.
        if len(set(ens.slots)) != len(ens.slots):
            for rep in range(N_REPS):
                for problem_id in range(len(dataset_sample)):
                    draw = draws_by_key[(ens.ensemble_id, dataset_key, rep, problem_id)]
                    gen_ids_by_model = {}
                    for model_key, gen_id in draw:
                        gen_ids_by_model.setdefault(model_key, []).append(gen_id)
                    for model_key, gen_ids in gen_ids_by_model.items():
                        if len(gen_ids) > 1:
                            assert len(set(gen_ids)) == len(gen_ids), (
                                f"duplicated model {model_key} drew non-distinct gen_ids: {gen_ids}"
                            )

        # Re-sampling the exact same (seed, dataset, ensemble_id, problem_id, rep) key
        # is byte-identical.
        redraw = sample_slots(ens, dataset=dataset_key, problem_id=0, rep=0)
        assert redraw == draws_by_key[(ens.ensemble_id, dataset_key, 0, 0)], (
            "sample_slots is not deterministic for the same key"
        )

        # rep 0 != rep 1 for at least one problem (independent draws).
        differs = any(
            draws_by_key[(ens.ensemble_id, dataset_key, 0, p)]
            != draws_by_key[(ens.ensemble_id, dataset_key, 1, p)]
            for p in range(len(dataset_sample))
        )
        assert differs, f"rep 0 and rep 1 draws were identical for every problem ({ens.ensemble_id}/{dataset_key})"

        # Seed reuse: sample_slots(rep=0) is byte-identical whether called "from the MAJ
        # loop" (the draws_by_key sweep above) or as a standalone call — the contract
        # BoN-MAV depends on to recompute MAJ's own rep-0 draw independently.
        standalone = sample_slots(ens, dataset=dataset_key, problem_id=1, rep=0)
        assert standalone == draws_by_key[(ens.ensemble_id, dataset_key, 0, 1)], (
            "sample_slots(rep=0) differs between call sites"
        )

        print(f"  ✅ {ens.ensemble_id}/{dataset_key}: sampler + seed-reuse invariants hold")

print("\n✅ Stage A sampler + seed-reuse checks complete.")


  SMOKE STAGE A — Majority Voting from the SC Stores (replay, no GPU)

----------------------------------------------------------------------
  Sampler + seed-reuse invariants
----------------------------------------------------------------------
  ✅ E1/math500: sampler + seed-reuse invariants hold
  ✅ E1/mmlu: sampler + seed-reuse invariants hold
  ✅ E4/math500: sampler + seed-reuse invariants hold
  ✅ E4/mmlu: sampler + seed-reuse invariants hold

✅ Stage A sampler + seed-reuse checks complete.


In [13]:
print(f"\n{'-'*70}\n  Replay fidelity, determinism, and metric composition\n{'-'*70}")

for ens in STAGE_A_ENSEMBLES:
    for dataset_key, dataset_sample, spec in STAGE_A_DATASETS:
        problem_id = 0
        rep = 0
        slots = sample_slots(ens, dataset=dataset_key, problem_id=problem_id, rep=rep)
        agent_ids = slot_agent_ids(ens)
        specs = [
            AgentSpec(
                agent_id=aid, model_key=mk, model_id=MODELS[mk]["model_id"],
                mode="replay", store=downloaded_stores[(mk, dataset_key)], gen_id=gid,
            )
            for aid, (mk, gid) in zip(agent_ids, slots)
        ]
        state = {
            "task": spec.render_task(dataset_sample[problem_id]), "problem_id": problem_id,
            "dataset": dataset_key, "samples": [], "vote": None, "metrics": None,
        }

        # Determinism: two independently, freshly seeded graph builds/invokes on the
        # same problem agree bit-for-bit — the same "two independent fresh seeds" check
        # section 9 already runs for run_self_consistency.
        graph1 = build_majority_voting_graph(specs, spec, rng=random.Random(TIE_BREAK_SEED))
        graph2 = build_majority_voting_graph(specs, spec, rng=random.Random(TIE_BREAK_SEED))
        out1 = graph1.invoke(dict(state))
        out2 = graph2.invoke(dict(state))
        assert out1["vote"] == out2["vote"], f"vote not reproducible for {ens.ensemble_id}/{dataset_key}"

        # Replay fidelity: each emitted AgentSample is byte-identical to the downloaded
        # store's record at that slot.
        for sample, (mk, gid) in zip(out1["samples"], slots):
            record = downloaded_stores[(mk, dataset_key)][(problem_id, gid)]
            assert sample["answer"] == record["answer"], "replay answer mismatch"
            assert sample["latency_s"] == record["latency_s"], "replay latency_s mismatch"
            assert sample["gpu_peak_mb"] == record["gpu_peak_mb"], "replay gpu_peak_mb mismatch"
            assert sample["model_key"] == mk

        # Composition: matches manual sums, and gpu_peak_mb agrees exactly with the
        # graph's own aggregate-node metrics (same function, same samples). Only
        # tokens_per_s is rule-dependent, so that's where heterogeneous ensembles
        # should diverge from the graph's default.
        composed = compose_generation_metrics(out1["samples"], tokens_per_s_rule="mean")
        manual_latency = sum(s["latency_s"] for s in out1["samples"])
        assert abs(composed["latency_s"] - manual_latency) < 1e-9

        peak_by_model = {}
        for s in out1["samples"]:
            peak_by_model[s["model_key"]] = max(peak_by_model.get(s["model_key"], 0.0), s["gpu_peak_mb"])
        manual_vram = sum(peak_by_model.values())
        assert abs(composed["gpu_peak_mb"] - manual_vram) < 1e-9
        assert composed["gpu_peak_mb"] == out1["metrics"]["gpu_peak_mb"], (
            "compose_generation_metrics' VRAM should match the graph's own "
            "aggregate-node metrics exactly"
        )

        is_heterogeneous = len(set(ens.slots)) > 1
        if is_heterogeneous:
            assert composed["tokens_per_s"] != out1["metrics"]["tokens_per_s"], (
                f"mean-rate tokens_per_s ({composed['tokens_per_s']:.2f}) should differ "
                f"from the graph's own recomputed-rate tokens_per_s "
                f"({out1['metrics']['tokens_per_s']:.2f}) for a heterogeneous ensemble "
                f"with distinct per-call rates"
            )

        print(f"  ✅ {ens.ensemble_id}/{dataset_key}: replay fidelity + determinism + "
              f"composition (VRAM={composed['gpu_peak_mb']:.0f} MB, "
              f"tokens/s mean={composed['tokens_per_s']:.2f} vs "
              f"recomputed={out1['metrics']['tokens_per_s']:.2f})")

print("\n✅ Stage A replay/determinism/composition checks complete.")


----------------------------------------------------------------------
  Replay fidelity, determinism, and metric composition
----------------------------------------------------------------------
  ✅ E1/math500: replay fidelity + determinism + composition (VRAM=6850 MB, tokens/s mean=15.59 vs recomputed=15.31)
  ✅ E1/mmlu: replay fidelity + determinism + composition (VRAM=6832 MB, tokens/s mean=12.65 vs recomputed=13.53)
  ✅ E4/math500: replay fidelity + determinism + composition (VRAM=9843 MB, tokens/s mean=14.34 vs recomputed=14.55)
  ✅ E4/mmlu: replay fidelity + determinism + composition (VRAM=9596 MB, tokens/s mean=13.47 vs recomputed=14.16)

✅ Stage A replay/determinism/composition checks complete.


In [14]:
print(f"\n{'-'*70}\n  Stage A closing run — one full MAJ-style W&B run (E1, MATH500)\n{'-'*70}")

import tempfile

stage_a_ens = E1
stage_a_dataset_key = "math500"
stage_a_sample = math_smoke_sample
stage_a_spec = MATH_ADAPTER["spec"]
stage_a_n_problems = len(stage_a_sample)
stage_a_agent_ids = slot_agent_ids(stage_a_ens)

stage_a_run = init_wandb(
    run_name=f"l1-maj{len(stage_a_ens.slots)}-{stage_a_ens.ensemble_id}-{stage_a_dataset_key}-smoke",
    project=PROJECT,
    config={
        "protocol": f"MAJ@{len(stage_a_ens.slots)}", "ensemble": stage_a_ens.ensemble_id,
        "ensemble_slots": list(stage_a_ens.slots), "slot_sample_seed": SLOT_SAMPLE_SEED,
        "tie_break_seed": TIE_BREAK_SEED, "n_reps": N_REPS, "eval_size": stage_a_n_problems,
        "data_seed": SMOKE_DATA_SEED,
    },
    tags=["l1", "majority-voting", "smoke", stage_a_dataset_key],
)
define_wandb_chart_only_metrics(prefixes=[f"{stage_a_dataset_key}_sample/*"])
stage_a_model_keys = list(dict.fromkeys(stage_a_ens.slots))  # distinct, first-appearance order
# Each stage's own W&B run fetches its own copy of every producer artifact it
# depends on, via load_generations_artifact(run=stage_a_run) — real
# producer->consumer lineage recorded on THIS run, not a bare use_artifact
# call plus reuse of the earlier round-trip run's downloads.
stage_a_stores, stage_a_alignment_counts = fetch_and_validate_stores(
    stage_a_model_keys, lambda mk: producer_refs[(mk, stage_a_dataset_key)],
    run=stage_a_run, dataset_sample=stage_a_sample, render_task=stage_a_spec.render_task,
    expected_dataset=stage_a_dataset_key, expected_data_seed=SMOKE_DATA_SEED,
    expected_eval_size=stage_a_n_problems, n_generators=len(SC_SEEDS),
)

correctness_matrix = np.zeros((stage_a_n_problems, N_REPS), dtype=int)
time_matrix = np.zeros((stage_a_n_problems, N_REPS), dtype=float)
gpu_peak_matrix = np.zeros((stage_a_n_problems, N_REPS), dtype=float)

results_columns = ["dataset", "ensemble", "protocol", "problem_id", "rep", "slots",
                    "task", "ground_truth", "final_output", "correct", "pct_votes_top",
                    "tie", "n_abstain", "latency_s", "gpu_peak_mb"]
results_rows = []
maj_tie_rng = random.Random(TIE_BREAK_SEED)

for rep in range(N_REPS):
    for problem_id, item in enumerate(stage_a_sample):
        slots = sample_slots(stage_a_ens, dataset=stage_a_dataset_key, problem_id=problem_id, rep=rep)
        specs = [
            AgentSpec(agent_id=aid, model_key=mk, model_id=MODELS[mk]["model_id"], mode="replay",
                      store=stage_a_stores[mk], gen_id=gid)
            for aid, (mk, gid) in zip(stage_a_agent_ids, slots)
        ]
        graph = build_majority_voting_graph(specs, stage_a_spec, rng=maj_tie_rng)
        task = stage_a_spec.render_task(item)
        ground_truth = stage_a_spec.get_truth(item)
        out = graph.invoke({
            "task": task, "problem_id": problem_id, "dataset": stage_a_dataset_key,
            "samples": [], "vote": None, "metrics": None,
        })
        vote = out["vote"]

        if vote["winner_idx"] is not None:
            winner_sample = out["samples"][vote["winner_idx"]]
            compare_value = (
                winner_sample["answer"] if stage_a_spec.compare_on == "answer" else winner_sample["parsed_output"]
            )
            correct = stage_a_spec.score(compare_value, ground_truth)
            winner_output = vote["winner_pred"]
        else:
            correct = False
            winner_output = None

        metrics = compose_generation_metrics(out["samples"], tokens_per_s_rule="mean")

        log_metrics({
            f"{stage_a_dataset_key}_sample/accuracy": int(correct),
            f"{stage_a_dataset_key}_sample/latency_s": round(metrics["latency_s"], 2),
            f"{stage_a_dataset_key}_sample/output_tokens": metrics["output_tokens"],
            f"{stage_a_dataset_key}_sample/total_tokens": metrics["total_tokens"],
            f"{stage_a_dataset_key}_sample/gpu_peak_mb": round(metrics["gpu_peak_mb"], 2),
            f"{stage_a_dataset_key}_sample/throughput_tok_s": round(metrics["tokens_per_s"], 4),
        })

        correctness_matrix[problem_id, rep] = int(correct)
        time_matrix[problem_id, rep] = metrics["latency_s"]
        gpu_peak_matrix[problem_id, rep] = metrics["gpu_peak_mb"]

        slots_str = ",".join(f"{mk}:{gid}" for mk, gid in slots)
        results_rows.append([
            stage_a_dataset_key, stage_a_ens.ensemble_id, f"MAJ@{len(stage_a_ens.slots)}", problem_id, rep,
            slots_str, task, ground_truth, winner_output, correct,
            round(vote["pct_votes_top"], 4), vote["tie"], vote["n_abstain"],
            round(metrics["latency_s"], 2), round(metrics["gpu_peak_mb"], 2),
        ])

results_table = wandb.Table(columns=results_columns, data=results_rows)
wandb.log({f"results_comparision_MATH500": results_table})

acc_stats = accuracy_statistics(correctness_matrix)
wandb.summary["accuracy"] = round(acc_stats["p_hat"], 6)
log_stat_summary("accuracy_stats", acc_stats, keys=["ci_lower", "ci_upper"])
for rep in range(N_REPS):
    rep_stats = accuracy_statistics(correctness_matrix[:, [rep]])
    log_stat_summary(f"accuracy_stats_rep{rep}", rep_stats, keys=["ci_lower", "ci_upper"])
    wandb.summary[f"accuracy_rep{rep}"] = round(rep_stats["p_hat"], 6)

artifact = wandb.Artifact(
    f"evaluation-matrices-maj{len(stage_a_ens.slots)}-{stage_a_ens.ensemble_id}-{stage_a_dataset_key}-smoke",
    type="evaluation-data",
    description=f"Stage A smoke MAJ evaluation matrices ({stage_a_n_problems} x {N_REPS}).",
)
with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
    np.savez(
        f.name,
        problem_ids=np.arange(stage_a_n_problems),
        maj_correctness=correctness_matrix,
        maj_times=time_matrix,
        maj_gpu_peak=gpu_peak_matrix,
    )
    artifact.add_file(f.name, name="evaluation_matrices.npz")
wandb.log_artifact(artifact)

finish_wandb()
print(f"✅ Stage A closing MAJ run complete: accuracy={acc_stats['p_hat']:.3f}")


----------------------------------------------------------------------
  Stage A closing run — one full MAJ-style W&B run (E1, MATH500)
----------------------------------------------------------------------


wandb:   2 of 2 files downloaded.  
wandb:   2 of 2 files downloaded.  
wandb:   2 of 2 files downloaded.  


math500_sample/accuracy,▁▁▁▁
math500_sample/gpu_peak_mb,▁█▆█
math500_sample/latency_s,▁██▇
math500_sample/output_tokens,▁██▇
math500_sample/throughput_tok_s,▆▅▁█
math500_sample/total_tokens,▁██▇
accuracy,1
accuracy_rep0,1
accuracy_rep1,1
accuracy_stats/ci_lower,1
accuracy_stats/ci_upper,1


✅ Stage A closing MAJ run complete: accuracy=1.000


## 11. Smoke Stage B — the Whole MAJ Protocol Live (GPU)

Validates §5.2: the scientific claim that **replaying SC generations is equivalent to running MAJ for real** — not just a mechanical check. MATH-500 only (the harder, `math_verify`-backed grouping path), the 2 smoke problems, 2 reps. Loads `E1`'s 3 distinct models co-resident, generates every slot live (mirroring the SC generation pass's `make_generate_fn`/`timed_invoke`, each slot seeded with `SC_SEEDS[sampled gen_id]` so the live draw mirrors the replayed one), asserts the co-resident VRAM budget, then repeats for `E4`'s 4 distinct models. Reports a live-vs-replay answer match rate (diagnostic, not asserted — CUDA non-determinism makes a hard equality assert flaky) and validates the §2.4 VRAM reconstruction rule against the raw co-resident peak.

In [15]:
print(f"\n{'='*70}\n  SMOKE STAGE B — Live Majority Voting (GPU) — MATH-500 only\n{'='*70}")

from utils.agents import build_hf_langchain_llm


def load_ensemble_models(ens):
    """Load every DISTINCT model in `ens` co-resident; returns {model_key: (model, tokenizer, templates)}."""
    loaded = {}
    for model_key in dict.fromkeys(ens.slots):
        model_cfg = MODELS[model_key]
        model, tokenizer = load_model_and_tokenizer(model_cfg["model_id"], quantize_4bit=True)
        templates = build_templates(no_think=model_cfg["no_think"])
        loaded[model_key] = (model, tokenizer, templates)
        print(f"  ✅ loaded {model_key} co-resident")
    return loaded


def unload_ensemble_models(loaded):
    for model_key in list(loaded):
        model, tokenizer, _ = loaded.pop(model_key)
        del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


def run_live_maj_stage(ens, dataset_key, dataset_sample, adapter, reps=(0, 1)):
    """Live-generate every slot for `reps` x dataset_sample via a live AgentSpec per
    slot, built the same way the SC generation pass builds one (make_generate_fn +
    timed_invoke) — proving replaying SC generations is equivalent to running MAJ for
    real. Returns {model_key: [per-call gpu_peak_mb, ...]} for the VRAM validation cell."""
    spec = adapter["spec"]
    loaded = load_ensemble_models(ens)

    allocated_mb = torch.cuda.memory_allocated() / 1e6
    print(f"  VRAM after loading {ens.ensemble_id}'s {len(loaded)} distinct models: {allocated_mb:.1f} MB")
    assert allocated_mb < 16000, f"co-resident VRAM budget exceeded: {allocated_mb:.1f} MB"

    agent_ids = slot_agent_ids(ens)
    live_tie_rng = random.Random(TIE_BREAK_SEED)
    match_count = 0
    total_count = 0
    live_peaks_by_model = {}

    for rep in reps:
        for problem_id, item in enumerate(dataset_sample):
            slots = sample_slots(ens, dataset=dataset_key, problem_id=problem_id, rep=rep)
            task = spec.render_task(item)

            live_specs = []
            for aid, (model_key, gen_id) in zip(agent_ids, slots):
                model, tokenizer, templates = loaded[model_key]
                seed = SC_SEEDS[gen_id]
                template = templates[dataset_key]
                llm = build_hf_langchain_llm(
                    model=model, tokenizer=tokenizer, max_new_tokens=adapter["max_new_tokens"],
                    temperature=GEN_COMMON["temperature"], top_p=GEN_COMMON["top_p"], top_k=GEN_COMMON["top_k"],
                    return_full_text=False, seed=seed,
                )
                chain = template | llm | StrOutputParser()
                generate_fn = make_generate_fn(
                    chain, template, model_key, MODELS[model_key]["model_id"],
                    aid, gen_id, spec.extract, tokenizer,
                )
                live_specs.append(AgentSpec(
                    agent_id=aid, model_key=model_key, model_id=MODELS[model_key]["model_id"],
                    mode="live", generate=generate_fn, seed=seed,
                ))

            graph = build_majority_voting_graph(live_specs, spec, rng=live_tie_rng)
            out = graph.invoke({
                "task": task, "problem_id": problem_id, "dataset": dataset_key,
                "samples": [], "vote": None, "metrics": None,
            })

            for sample, (model_key, gen_id) in zip(out["samples"], slots):
                live_peaks_by_model.setdefault(model_key, []).append(sample["gpu_peak_mb"])
                sc_record = downloaded_stores[(model_key, dataset_key)][(problem_id, gen_id)]
                total_count += 1
                if sample["answer"] == sc_record["answer"]:
                    match_count += 1

            print(f"  rep={rep} problem={problem_id}: vote winner_idx={out['vote']['winner_idx']} "
                  f"tie={out['vote']['tie']}")


    del live_specs, graph, out, chain, llm, generate_fn, model, tokenizer
    unload_ensemble_models(loaded)
    final_allocated_mb = torch.cuda.memory_allocated() / 1e6
    print(f"  VRAM after unloading {ens.ensemble_id}: {final_allocated_mb:.1f} MB")
    assert final_allocated_mb < 1000, f"VRAM not released after unloading {ens.ensemble_id}: {final_allocated_mb:.1f} MB"

    match_rate = match_count / total_count if total_count else 0.0
    print(f"  Diagnostic: live == SC-record answer match rate = {match_rate:.1%} ({match_count}/{total_count})")
    if match_rate < 1.0:
        print("  ⚠️  match rate < 100% — CUDA non-determinism (expected to some degree; not asserted)")

    return live_peaks_by_model


stage_b_dataset_key = "math500"
stage_b_sample = math_smoke_sample
peaks_e1 = run_live_maj_stage(E1, stage_b_dataset_key, stage_b_sample, MATH_ADAPTER)
peaks_e4 = run_live_maj_stage(E4, stage_b_dataset_key, stage_b_sample, MATH_ADAPTER)

print("\n✅ Stage B live generation complete for E1 and E4.")


  SMOKE STAGE B — Live Majority Voting (GPU) — MATH-500 only


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

  ✅ loaded qwen3-1.7b co-resident


Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

  ✅ loaded ministral-3-3b co-resident


...

[transformers] The model 'Mistral3ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4F

  rep=0 problem=1: vote winner_idx=0 tie=False


[transformers] The model 'Mistral3ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4F

  rep=1 problem=0: vote winner_idx=0 tie=False
  rep=1 problem=1: vote winner_idx=0 tie=False
  VRAM after unloading E1: 6648.0 MB


AssertionError: VRAM not released after unloading E1: 6648.0 MB

In [ ]:
print(f"\n{'-'*70}\n  VRAM rule validation — co-resident peak vs §2.4 reconstruction\n{'-'*70}")


def validate_vram_reconstruction(ens, dataset_key, peaks_by_model, tol=0.15):
    """Compare the raw measured co-resident peak (max gpu_peak_mb across every live
    call in this ensemble's run, device-wide since every model shared the same CUDA
    device during Stage B) against the §2.4 reconstruction: the sum, over each
    distinct model in the ensemble, of that model's own single-model SC peak (max
    gpu_peak_mb across its downloaded smoke-store records). The live co-resident
    peaks must NOT be fed back through compose_generation_metrics as if they were
    single-model SC peaks — that is exactly the caveat §2.4 flags; this cell instead
    reports and compares the two numbers."""
    raw_coresident_peak = max(p for peaks in peaks_by_model.values() for p in peaks)

    reconstruction = 0.0
    for model_key in dict.fromkeys(ens.slots):
        sc_peaks = [r["gpu_peak_mb"] for r in downloaded_stores[(model_key, dataset_key)].values()]
        reconstruction += max(sc_peaks)

    rel_diff = abs(raw_coresident_peak - reconstruction) / reconstruction if reconstruction else float("inf")
    print(f"  {ens.ensemble_id}: raw co-resident peak={raw_coresident_peak:.0f} MB, "
          f"§2.4 reconstruction={reconstruction:.0f} MB, rel diff={rel_diff:.1%}")
    assert rel_diff <= tol, (
        f"{ens.ensemble_id}: co-resident VRAM ({raw_coresident_peak:.0f} MB) vs reconstruction "
        f"({reconstruction:.0f} MB) diverge by {rel_diff:.1%}, exceeding the {tol:.0%} tolerance"
    )


validate_vram_reconstruction(E1, stage_b_dataset_key, peaks_e1)
validate_vram_reconstruction(E4, stage_b_dataset_key, peaks_e4)

print("\n✅ Stage B VRAM rule validation complete.")

## 12. Smoke Stage C — BoN-MAV Mini-Run (GPU)

Validates §5.3: loads `qwen3-1.7b` as the sole live verifier, candidates are `E2`'s 3 models replayed from `downloaded_stores` at `rep=0`, and exercises **all 3 `max_new_tokens` strategy tiers** from `MATH-S1` (AV1 `direct`, AV2 `reasoning`, AV4 `step_by_step`) — every branch `verify_fn` can take, before any real `v5.x` session commits to it. Logs verdict parse rate, per-aspect approval rate, and selection. This stage doubles as the §6 GPU-budget calibration point: it logs each aspect's *actual* output token count next to its cap, the first real (if tiny) data point for whether `direct`/`reasoning`/`step_by_step` terminate near their cap or much earlier.

In [ ]:
print(f"\n{'='*70}\n  SMOKE STAGE C — BoN-MAV Mini-Run (GPU)\n{'='*70}")

from utils.ensembles import E2
from utils.graphs import build_bon_mav_graph
from utils.mav import (
    MAV_ASPECTS, MAV_ASPECT_SUBSETS, STRATEGY_MAX_NEW_TOKENS,
    approval_diagnostics, parse_verdict, select_bon_candidate,
)

stage_c_dataset_key = "math500"
stage_c_sample = math_smoke_sample
stage_c_verifier_key = "qwen3-1.7b"
stage_c_spec = MATH_ADAPTER["spec"]

# All 3 strategy tiers from MATH-S1 (AV1 direct, AV2 reasoning, AV4 step_by_step) —
# AV3 (also reasoning) is skipped, since the point is to exercise every distinct
# max_new_tokens branch of verify_fn at least once, not the full subset.
stage_c_aspect_ids = [
    MAV_ASPECT_SUBSETS["math500"]["S1"][0],  # av1 - direct
    MAV_ASPECT_SUBSETS["math500"]["S1"][1],  # av2 - reasoning
    MAV_ASPECT_SUBSETS["math500"]["S1"][3],  # av4 - step_by_step
]
print(f"  Aspects exercised (1 per strategy tier): {stage_c_aspect_ids}")


def make_verify_fn(model, tokenizer, model_key, dataset_key, sampling):
    """A timed_invoke-based closure over the (single, already-loaded) verifier
    model, dispatching on the aspect's strategy for max_new_tokens — identical
    instrumentation to the SC generation pass's make_generate_fn, but templated
    per-aspect instead of per-benchmark."""
    llm_by_max_tokens = {}

    def get_llm(max_new_tokens):
        if max_new_tokens not in llm_by_max_tokens:
            llm_by_max_tokens[max_new_tokens] = build_hf_langchain_llm(
                model=model, tokenizer=tokenizer, max_new_tokens=max_new_tokens,
                temperature=sampling["temperature"], top_p=sampling["top_p"], top_k=sampling["top_k"],
                return_full_text=False,
            )
        return llm_by_max_tokens[max_new_tokens]

    def verify(task, candidate_answer, aspect_id):
        aspect = MAV_ASPECTS[dataset_key][aspect_id]
        prompt_template = PromptTemplate.from_template(aspect["prompt"])
        llm = get_llm(aspect["max_new_tokens"])
        chain = prompt_template | llm | StrOutputParser()
        r = timed_invoke(chain, prompt_template, tokenizer, {"task": task, "candidate": candidate_answer})
        return {
            "candidate_idx": -1, "aspect_id": aspect_id, "verdict": parse_verdict(r["output_text"]),
            "raw_output": r["output_text"], "latency_s": r["latency_s"],
            "input_tokens": r["input_tokens"], "output_tokens": r["output_tokens"],
            "total_tokens": r["total_tokens"], "gpu_peak_mb": r["gpu_peak_mb"],
            "model_key": model_key,
        }
    return verify


def make_select_fn(spec):
    def select_fn(n_candidates, approvals, candidate_answers, *, rng):
        return select_bon_candidate(n_candidates, approvals, candidate_answers, spec=spec, rng=rng)
    return select_fn


verifier_cfg = MODELS[stage_c_verifier_key]
verifier_model, verifier_tokenizer = load_model_and_tokenizer(verifier_cfg["model_id"], quantize_4bit=True)
warmup_model(verifier_model, verifier_tokenizer, max_new_tokens=64, n_iterations=1,
             temperature=GEN_COMMON["temperature"], top_p=GEN_COMMON["top_p"], top_k=GEN_COMMON["top_k"])
print(f"  ✅ verifier {stage_c_verifier_key} loaded")

stage_c_run = init_wandb(
    run_name="l1-bonmav-stage-c-smoke", project=PROJECT,
    config={
        "protocol": "BoN-MAV@3/M3", "verifier_key": stage_c_verifier_key, "candidate_ensemble": E2.ensemble_id,
        "aspect_ids": stage_c_aspect_ids, "eval_dataset": stage_c_dataset_key, "eval_size": len(stage_c_sample),
        "data_seed": SMOKE_DATA_SEED, "tie_break_seed": TIE_BREAK_SEED,
    },
    tags=["l1", "bon-mav", "smoke", stage_c_dataset_key],
)
define_wandb_chart_only_metrics(prefixes=[f"{stage_c_dataset_key}_sample/*"])
stage_c_model_keys = list(dict.fromkeys(E2.slots))
# Own W&B run fetches its own copy of every candidate producer artifact via
# load_generations_artifact(run=stage_c_run) — real lineage on THIS run.
stage_c_stores, stage_c_alignment_counts = fetch_and_validate_stores(
    stage_c_model_keys, lambda mk: producer_refs[(mk, stage_c_dataset_key)],
    run=stage_c_run, dataset_sample=stage_c_sample, render_task=stage_c_spec.render_task,
    expected_dataset=stage_c_dataset_key, expected_data_seed=SMOKE_DATA_SEED,
    expected_eval_size=len(stage_c_sample), n_generators=len(SC_SEEDS),
)

verify_fn = make_verify_fn(verifier_model, verifier_tokenizer, stage_c_verifier_key, stage_c_dataset_key, GEN_COMMON)
select_fn = make_select_fn(stage_c_spec)

bonmav_tie_rng = random.Random(TIE_BREAK_SEED)
e2_agent_ids = slot_agent_ids(E2)
all_approvals = []
all_selections = []
all_candidate_samples = []  # the "reused" side of the reused_gen_*/live_verify_* split

for problem_id, item in enumerate(stage_c_sample):
    slots = sample_slots(E2, dataset=stage_c_dataset_key, problem_id=problem_id, rep=0)
    candidate_specs = [
        AgentSpec(agent_id=aid, model_key=mk, model_id=MODELS[mk]["model_id"], mode="replay",
                  store=stage_c_stores[mk], gen_id=gid)
        for aid, (mk, gid) in zip(e2_agent_ids, slots)
    ]
    graph = build_bon_mav_graph(candidate_specs, stage_c_aspect_ids, verify_fn, select_fn, rng=bonmav_tie_rng)
    task = stage_c_spec.render_task(item)
    out = graph.invoke({
        "task": task, "problem_id": problem_id, "dataset": stage_c_dataset_key,
        "samples": [], "approvals": [], "selection": None,
    })

    ground_truth = stage_c_spec.get_truth(item)
    selection = out["selection"]
    selected_sample = out["samples"][selection["selected_idx"]]
    compare_value = selected_sample["answer"] if stage_c_spec.compare_on == "answer" else selected_sample["parsed_output"]
    correct = stage_c_spec.score(compare_value, ground_truth)
    composed = compose_generation_metrics(out["samples"] + out["approvals"], tokens_per_s_rule="mean")
    log_metrics({
        f"{stage_c_dataset_key}_sample/accuracy": int(correct),
        f"{stage_c_dataset_key}_sample/latency_s": round(composed["latency_s"], 2),
        f"{stage_c_dataset_key}_sample/output_tokens": composed["output_tokens"],
        f"{stage_c_dataset_key}_sample/total_tokens": composed["total_tokens"],
        f"{stage_c_dataset_key}_sample/gpu_peak_mb": round(composed["gpu_peak_mb"], 2),
        f"{stage_c_dataset_key}_sample/throughput_tok_s": round(composed["tokens_per_s"], 4),
    })

    all_approvals.extend(out["approvals"])
    all_selections.append(out["selection"])
    all_candidate_samples.extend(out["samples"])
    print(f"  problem={problem_id}: selected_idx={out['selection']['selected_idx']} "
          f"scores={out['selection']['scores']} tie={out['selection']['tie']} correct={correct}")

print(f"\n✅ Stage C BoN-MAV generation complete: {len(all_approvals)} approvals "
      f"({len(stage_c_sample)} problems x {len(e2_agent_ids)} candidates x {len(stage_c_aspect_ids)} aspects).")

In [ ]:
print(f"\n{'-'*70}\n  Verdict parse rate, per-aspect approval rate, and §6 calibration data\n{'-'*70}")

diag = approval_diagnostics(all_approvals, n_candidates=len(E2.slots))
for aspect_id, stats in diag.items():
    strategy = MAV_ASPECTS[stage_c_dataset_key][aspect_id]["strategy"]
    print(f"  {aspect_id} ({strategy}): approval_rate={stats['approval_rate']:.1%} "
          f"parse_rate={stats['parse_rate']:.1%} n={stats['n']}")
    wandb.summary[f"approval_rate/{aspect_id}"] = round(stats["approval_rate"], 4)
    wandb.summary[f"parse_rate/{aspect_id}"] = round(stats["parse_rate"], 4)

overall_parse_rate = sum(1 for a in all_approvals if a["verdict"] is not None) / len(all_approvals)
n_ties = sum(1 for s in all_selections if s["tie"])
wandb.summary["overall_parse_rate"] = round(overall_parse_rate, 4)
wandb.summary["selection_tie_rate"] = round(n_ties / len(all_selections), 4)
print(f"\n  Overall verdict parse rate: {overall_parse_rate:.1%}")
print(f"  Selection tie rate: {n_ties}/{len(all_selections)}")

# §6 GPU-budget calibration point (see the ## 12 markdown above for why this matters).
tokens_by_strategy = {}
for approval in all_approvals:
    strategy = MAV_ASPECTS[stage_c_dataset_key][approval["aspect_id"]]["strategy"]
    tokens_by_strategy.setdefault(strategy, []).append(approval["output_tokens"])

for strategy, tokens_list in tokens_by_strategy.items():
    cap = STRATEGY_MAX_NEW_TOKENS[strategy]
    mean_tokens = sum(tokens_list) / len(tokens_list)
    frac_of_cap = mean_tokens / cap
    print(f"  {strategy}: mean_output_tokens={mean_tokens:.0f} / cap={cap} "
          f"({frac_of_cap:.0%} of cap, n={len(tokens_list)})")
    wandb.summary[f"calibration/{strategy}_mean_output_tokens"] = round(mean_tokens, 1)
    wandb.summary[f"calibration/{strategy}_frac_of_cap"] = round(frac_of_cap, 4)

# Verifier input_tokens context-length sanity (~3k expected: aspect prompt + question
# + a full candidate answer, reprocessed from scratch on every independent call).
mean_input_tokens = sum(a["input_tokens"] for a in all_approvals) / len(all_approvals)
wandb.summary["mean_input_tokens"] = round(mean_input_tokens, 1)
print(f"  Mean verifier input_tokens: {mean_input_tokens:.0f} (context-length sanity, expect ~3k at full scale)")

# reused_gen_* / live_verify_* cost split (§2.5) — only the verification half is
# fresh compute; the candidate half is entirely replayed from the SC artifacts.
reused_gen_metrics = compose_generation_metrics(all_candidate_samples, tokens_per_s_rule="mean")
live_verify_metrics = compose_generation_metrics(all_approvals, tokens_per_s_rule="mean")
wandb.summary["reused_gen_latency_s"] = round(reused_gen_metrics["latency_s"], 2)
wandb.summary["reused_gen_total_tokens"] = reused_gen_metrics["total_tokens"]
wandb.summary["live_verify_latency_s"] = round(live_verify_metrics["latency_s"], 2)
wandb.summary["live_verify_total_tokens"] = live_verify_metrics["total_tokens"]
print(f"  reused_gen: latency={reused_gen_metrics['latency_s']:.2f}s tokens={reused_gen_metrics['total_tokens']} "
      f"| live_verify: latency={live_verify_metrics['latency_s']:.2f}s tokens={live_verify_metrics['total_tokens']}")

del verifier_model, verifier_tokenizer, verify_fn, graph, out
gc.collect()
torch.cuda.empty_cache()
final_allocated_mb = torch.cuda.memory_allocated() / 1e6
print(f"\n  VRAM after unloading verifier: {final_allocated_mb:.1f} MB")
assert final_allocated_mb < 1000, f"VRAM not released after unloading verifier: {final_allocated_mb:.1f} MB"

finish_wandb()
print("\n✅ Stage C diagnostics + calibration logging complete.")

## 13. Smoke Stage C2 — BoN-MAV Without SC-Reuse (GPU)

Stage C above only ever replays SC-persisted candidates; the verifier is the only live model there. This stage closes that gap: `E2`'s 3 candidate models generate **live** (not replayed) alongside a freshly-loaded verifier, proving `build_bon_mav_graph` works unmodified with `mode="live"` candidate specs — the same live/replay duality `utils.graphs.AgentSpec` already exercises for Majority Voting in Stage B. The same 3 strategy-tier aspects and the same `E2` rep-0 slot draw as Stage C are reused here for direct comparability, and the diagnostics cell logs a full-live cost total against Stage C's `reused_gen_*`/`live_verify_*` split — the concrete quantified evidence for how much SC-reuse saves.

In [ ]:
print(f"\n{'='*70}\n  SMOKE STAGE C2 — BoN-MAV Without SC-Reuse (GPU) — MATH-500 only\n{'='*70}")

stage_c2_dataset_key = "math500"
stage_c2_sample = math_smoke_sample
stage_c2_verifier_key = stage_c_verifier_key  # qwen3-1.7b — same verifier as Stage C
stage_c2_spec = MATH_ADAPTER["spec"]
stage_c2_aspect_ids = stage_c_aspect_ids  # same 3 strategy tiers as Stage C

e2_loaded = load_ensemble_models(E2)  # co-resident candidates, reusing Stage B's loader

verifier_cfg_c2 = MODELS[stage_c2_verifier_key]
c2_verifier_model, c2_verifier_tokenizer = load_model_and_tokenizer(verifier_cfg_c2["model_id"], quantize_4bit=True)
warmup_model(c2_verifier_model, c2_verifier_tokenizer, max_new_tokens=64, n_iterations=1,
             temperature=GEN_COMMON["temperature"], top_p=GEN_COMMON["top_p"], top_k=GEN_COMMON["top_k"])
print(f"  ✅ verifier {stage_c2_verifier_key} loaded co-resident with E2's {len(e2_loaded)} candidate models")

allocated_mb = torch.cuda.memory_allocated() / 1e6
print(f"  VRAM after loading E2 + verifier ({len(e2_loaded) + 1} distinct models): {allocated_mb:.1f} MB")
assert allocated_mb < 16000, f"co-resident VRAM budget exceeded: {allocated_mb:.1f} MB"

verify_fn_c2 = make_verify_fn(c2_verifier_model, c2_verifier_tokenizer, stage_c2_verifier_key,
                               stage_c2_dataset_key, GEN_COMMON)
select_fn_c2 = make_select_fn(stage_c2_spec)

stage_c2_run = init_wandb(
    run_name="l1-bonmav-stage-c2-live-smoke", project=PROJECT,
    config={
        "protocol": "BoN-MAV@3/M3-live", "verifier_key": stage_c2_verifier_key,
        "candidate_ensemble": E2.ensemble_id, "aspect_ids": stage_c2_aspect_ids,
        "eval_dataset": stage_c2_dataset_key, "eval_size": len(stage_c2_sample),
        "data_seed": SMOKE_DATA_SEED, "tie_break_seed": TIE_BREAK_SEED,
    },
    tags=["l1", "bon-mav", "smoke", "live-candidates", stage_c2_dataset_key],
)
define_wandb_chart_only_metrics(prefixes=[f"{stage_c2_dataset_key}_sample/*"])
stage_c2_model_keys = list(dict.fromkeys(E2.slots))
# Own W&B run fetches its own copy of every candidate producer artifact via
# load_generations_artifact(run=stage_c2_run) — real lineage on THIS run, and
# the store used for the live-vs-SC-record diagnostic below.
stage_c2_stores, stage_c2_alignment_counts = fetch_and_validate_stores(
    stage_c2_model_keys, lambda mk: producer_refs[(mk, stage_c2_dataset_key)],
    run=stage_c2_run, dataset_sample=stage_c2_sample, render_task=stage_c2_spec.render_task,
    expected_dataset=stage_c2_dataset_key, expected_data_seed=SMOKE_DATA_SEED,
    expected_eval_size=len(stage_c2_sample), n_generators=len(SC_SEEDS),
)

c2_tie_rng = random.Random(TIE_BREAK_SEED)
all_c2_approvals = []
all_c2_selections = []
all_c2_candidate_samples = []
c2_match_count = 0
c2_total_count = 0

for problem_id, item in enumerate(stage_c2_sample):
    slots = sample_slots(E2, dataset=stage_c2_dataset_key, problem_id=problem_id, rep=0)  # same draw Stage C used
    task = stage_c2_spec.render_task(item)

    live_candidate_specs = []
    for aid, (model_key, gen_id) in zip(e2_agent_ids, slots):
        model, tokenizer, templates = e2_loaded[model_key]
        seed = SC_SEEDS[gen_id]
        template = templates[stage_c2_dataset_key]
        llm = build_hf_langchain_llm(
            model=model, tokenizer=tokenizer, max_new_tokens=MATH_ADAPTER["max_new_tokens"],
            temperature=GEN_COMMON["temperature"], top_p=GEN_COMMON["top_p"], top_k=GEN_COMMON["top_k"],
            return_full_text=False, seed=seed,
        )
        chain = template | llm | StrOutputParser()
        generate_fn = make_generate_fn(
            chain, template, model_key, MODELS[model_key]["model_id"],
            aid, gen_id, stage_c2_spec.extract, tokenizer,
        )
        live_candidate_specs.append(AgentSpec(
            agent_id=aid, model_key=model_key, model_id=MODELS[model_key]["model_id"],
            mode="live", generate=generate_fn, seed=seed,
        ))

    graph = build_bon_mav_graph(live_candidate_specs, stage_c2_aspect_ids, verify_fn_c2, select_fn_c2, rng=c2_tie_rng)
    out = graph.invoke({
        "task": task, "problem_id": problem_id, "dataset": stage_c2_dataset_key,
        "samples": [], "approvals": [], "selection": None,
    })

    ground_truth = stage_c2_spec.get_truth(item)
    selection = out["selection"]
    selected_sample = out["samples"][selection["selected_idx"]]
    compare_value = selected_sample["answer"] if stage_c2_spec.compare_on == "answer" else selected_sample["parsed_output"]
    correct = stage_c2_spec.score(compare_value, ground_truth)
    composed = compose_generation_metrics(out["samples"] + out["approvals"], tokens_per_s_rule="mean")
    log_metrics({
        f"{stage_c2_dataset_key}_sample/accuracy": int(correct),
        f"{stage_c2_dataset_key}_sample/latency_s": round(composed["latency_s"], 2),
        f"{stage_c2_dataset_key}_sample/output_tokens": composed["output_tokens"],
        f"{stage_c2_dataset_key}_sample/total_tokens": composed["total_tokens"],
        f"{stage_c2_dataset_key}_sample/gpu_peak_mb": round(composed["gpu_peak_mb"], 2),
        f"{stage_c2_dataset_key}_sample/throughput_tok_s": round(composed["tokens_per_s"], 4),
    })

    all_c2_approvals.extend(out["approvals"])
    all_c2_selections.append(out["selection"])
    all_c2_candidate_samples.extend(out["samples"])

    for sample, (model_key, gen_id) in zip(out["samples"], slots):
        sc_record = stage_c2_stores[model_key][(problem_id, gen_id)]
        c2_total_count += 1
        if sample["answer"] == sc_record["answer"]:
            c2_match_count += 1

    print(f"  problem={problem_id}: selected_idx={out['selection']['selected_idx']} "
          f"scores={out['selection']['scores']} tie={out['selection']['tie']} correct={correct}")

print(f"\n✅ Stage C2 BoN-MAV live generation complete: {len(all_c2_approvals)} approvals "
      f"({len(stage_c2_sample)} problems x {len(e2_agent_ids)} candidates x {len(stage_c2_aspect_ids)} aspects).")

c2_match_rate = c2_match_count / c2_total_count if c2_total_count else 0.0
print(f"  Diagnostic: live == SC-record answer match rate = {c2_match_rate:.1%} ({c2_match_count}/{c2_total_count})")
if c2_match_rate < 1.0:
    print("  ⚠️  match rate < 100% — CUDA non-determinism (expected to some degree; not asserted)")

In [ ]:
print(f"\n{'-'*70}\n  Stage C2 — full-live cost vs Stage C's reused_gen_*/live_verify_* split\n{'-'*70}")

diag_c2 = approval_diagnostics(all_c2_approvals, n_candidates=len(E2.slots))
for aspect_id, stats in diag_c2.items():
    strategy = MAV_ASPECTS[stage_c2_dataset_key][aspect_id]["strategy"]
    print(f"  {aspect_id} ({strategy}): approval_rate={stats['approval_rate']:.1%} "
          f"parse_rate={stats['parse_rate']:.1%} n={stats['n']}")
    wandb.summary[f"approval_rate/{aspect_id}"] = round(stats["approval_rate"], 4)
    wandb.summary[f"parse_rate/{aspect_id}"] = round(stats["parse_rate"], 4)

c2_overall_parse_rate = sum(1 for a in all_c2_approvals if a["verdict"] is not None) / len(all_c2_approvals)
c2_n_ties = sum(1 for s in all_c2_selections if s["tie"])
wandb.summary["overall_parse_rate"] = round(c2_overall_parse_rate, 4)
wandb.summary["selection_tie_rate"] = round(c2_n_ties / len(all_c2_selections), 4)
print(f"  Overall verdict parse rate: {c2_overall_parse_rate:.1%}")
print(f"  Selection tie rate: {c2_n_ties}/{len(all_c2_selections)}")

# Full-live cost: candidates and verification are both fresh compute this time,
# unlike Stage C's reused_gen_*/live_verify_* split (candidates replayed there).
full_live_metrics = compose_generation_metrics(
    all_c2_candidate_samples + all_c2_approvals, tokens_per_s_rule="mean",
)
wandb.summary["full_live_latency_s"] = round(full_live_metrics["latency_s"], 2)
wandb.summary["full_live_total_tokens"] = full_live_metrics["total_tokens"]
print(f"  full_live: latency={full_live_metrics['latency_s']:.2f}s tokens={full_live_metrics['total_tokens']} "
      f"(vs Stage C reused_gen+live_verify: "
      f"latency={reused_gen_metrics['latency_s'] + live_verify_metrics['latency_s']:.2f}s "
      f"tokens={reused_gen_metrics['total_tokens'] + live_verify_metrics['total_tokens']})")

del graph, out, chain, llm, generate_fn, model, tokenizer, live_candidate_specs, verify_fn_c2
unload_ensemble_models(e2_loaded)
del c2_verifier_model, c2_verifier_tokenizer
gc.collect()
torch.cuda.empty_cache()
final_allocated_mb = torch.cuda.memory_allocated() / 1e6
print(f"\n  VRAM after unloading E2 + verifier: {final_allocated_mb:.1f} MB")
assert final_allocated_mb < 1000, f"VRAM not released after unloading Stage C2 models: {final_allocated_mb:.1f} MB"

finish_wandb()
print("\n✅ Stage C2 diagnostics + live-vs-reused cost comparison complete.")

## 14. Invariant Checklist

Hard asserts covering every claim this smoke test needs to establish before the 8 real SC notebooks, the `v4.x` Majority Voting notebooks, and the `v5.x` BoN-MAV notebooks run — including (§5.4) the MAJ/BoN-MAV invariants Stages A–C2 above depend on.

In [ ]:
print(f"\n{'='*70}\n  INVARIANT CHECKLIST\n{'='*70}\n")

from utils.persistence import assert_unique_pk

checks = []

# (a) generation count == 4 x 2 x 2 x 5 = 80
total_generations = sum(len(recs) for recs in in_memory_records.values())
expected_total = len(MODEL_KEYS) * len(DATASETS) * SMOKE_EVAL_SIZE * len(SC_SEEDS)
checks.append((
    f"generation count == {len(MODEL_KEYS)}x{len(DATASETS)}x{SMOKE_EVAL_SIZE}x{len(SC_SEEDS)} = {expected_total}",
    total_generations == expected_total, f"got {total_generations}",
))

# (b) SC@k ladder present for every (model, benchmark) with correct subset counts, SC@5 in [0,1]
expected_subset_counts = {1: 5, 3: 10, 4: 5, 5: 1}
ladder_ok = True
for key, k_results in sc_results.items():
    for k, expected_n in expected_subset_counts.items():
        if k not in k_results or k_results[k]["n_subsets"] != expected_n:
            ladder_ok = False
        elif not (0.0 <= k_results[k]["accuracy"] <= 1.0):
            ladder_ok = False
checks.append(("SC@k ladder present for every (model, benchmark) with subset counts 5/10/5/1, SC@5 in [0,1]",
                ladder_ok, ""))

# (c) artifact round-trip fidelity (re-asserted from the previous cell)
checks.append(("artifact round-trip fidelity (byte-identical `answer` after download)",
                fidelity_downloaded["answer"] == fidelity_original["answer"], ""))

# (d) alignment checks passed for all 8 producer artifacts
checks.append(("alignment checks passed for all 8 producer artifacts", not alignment_errors, str(alignment_errors)))

# (e) primary-key uniqueness across the merged 8-artifact record set — the
# on-Kaggle proof that (dataset, problem_id, model_key, gen_id) holds as a real
# PK once every model x benchmark's records are pooled together.
all_records = [r for recs in in_memory_records.values() for r in recs]
pk_error = None
try:
    assert_unique_pk(all_records)
except ValueError as e:
    pk_error = str(e)
checks.append(("primary key (dataset, problem_id, model_key, gen_id) unique across all 8 artifacts",
                pk_error is None, pk_error or ""))

# (f)+(g) determinism and metric composition — built directly off build_generation_store
# (not sliced from the gen_id-major raw records list). Two independent, freshly seeded
# runs of the exact production entry point (utils.voting.run_self_consistency) must
# agree bit-for-bit — the reproducibility property the whole SC ladder relies on.
det_model_key = MODEL_KEYS[0]
det_key = (det_model_key, "math500")
store_det = build_generation_store(in_memory_records[det_key])

r1 = run_self_consistency(store_det, math_smoke_sample, MATH_ADAPTER["spec"], [5],
                          n_generators=len(SC_SEEDS), rng=random.Random(TIE_BREAK_SEED),
                          model_key=det_model_key, model_id=MODELS[det_model_key]["model_id"])
r2 = run_self_consistency(store_det, math_smoke_sample, MATH_ADAPTER["spec"], [5],
                          n_generators=len(SC_SEEDS), rng=random.Random(TIE_BREAK_SEED),
                          model_key=det_model_key, model_id=MODELS[det_model_key]["model_id"])
checks.append(("vote determinism (identical run_self_consistency results from a fresh seed)", r1 == r2, ""))

subset_recs = [store_det[(0, g)] for g in range(len(SC_SEEDS))]  # problem_id=0, gen_id=0..4
manual_latency = sum(r["latency_s"] for r in subset_recs)
manual_peak    = max(r["gpu_peak_mb"] for r in subset_recs)
composed = compose_generation_metrics(subset_recs)
checks.append((
    "metric composition matches manual sums/max",
    abs(composed["latency_s"] - manual_latency) < 1e-6 and abs(composed["gpu_peak_mb"] - manual_peak) < 1e-6,
    f"composed={composed['latency_s']:.2f}/{composed['gpu_peak_mb']:.1f} vs "
    f"manual={manual_latency:.2f}/{manual_peak:.1f}",
))

# (h) VRAM bounds + release between models
all_peaks = [r["gpu_peak_mb"] for recs in in_memory_records.values() for r in recs]
final_allocated_mb = torch.cuda.memory_allocated() / 1e6
checks.append(("every recorded gpu_peak_mb < 16000 (16 GB budget)",
                all(p < 16000 for p in all_peaks), f"max={max(all_peaks):.0f} MB"))
checks.append(("VRAM released after the last model unload",
                final_allocated_mb < 1000, f"{final_allocated_mb:.1f} MB"))

# (i) lineage recorded: producer_ref captured for all 8 (model, dataset) pairs
checks.append(("lineage: producer_ref recorded for all 8 (model, dataset) pairs",
                len(producer_refs) == len(MODEL_KEYS) * len(DATASETS) and all(producer_refs.values()), ""))

# (j) extraction-failure / tie rate diagnostics (printed, not asserted — informational)
n_records_all = sum(len(recs) for recs in in_memory_records.values())
n_extraction_failures = sum(
    1 for recs in in_memory_records.values() for r in recs if r["parsed_output"] is None
)
extraction_failure_rate = n_extraction_failures / n_records_all if n_records_all else 0.0

print(f"Extraction-failure rate across all {n_records_all} generations: {extraction_failure_rate:.2%}\n")

# §5.4 Stage D additions: MAJ / BoN-MAV invariants from Stages A–C

# (k) BoN-MAV approval matrix shape == n_candidates x n_aspects, per problem.
n_candidates_expected = len(e2_agent_ids)
n_aspects_expected = len(stage_c_aspect_ids)
expected_approval_count = n_candidates_expected * n_aspects_expected * len(stage_c_sample)
checks.append((
    f"BoN-MAV approval matrix shape == n_candidates x n_aspects per problem "
    f"({n_candidates_expected}x{n_aspects_expected}, {len(stage_c_sample)} problems)",
    len(all_approvals) == expected_approval_count,
    f"got {len(all_approvals)}, expected {expected_approval_count}",
))

# (l) unparseable verdicts (verdict=None) count as non-approvals: no selection score
# can ever exceed the number of aspects (the maximum possible if every aspect
# approved), which only holds if None/False both contribute zero.
scores_bounded = all(s <= n_aspects_expected for sel in all_selections for s in sel["scores"])
checks.append(("unparseable verdicts scored as non-approval (every selection score <= n_aspects)",
                scores_bounded, ""))

# (m) selection deterministic on re-invoke from a fresh seed. Uses a scripted verify
# closure (the live verifier was already unloaded in Stage C) that returns an
# unconditional approval for every candidate/aspect — a guaranteed tie, so this
# specifically exercises the tie-break's own fresh-seed reproducibility.
def _scripted_verify_for_determinism_check(task, candidate_answer, aspect_id):
    return {
        "candidate_idx": -1, "aspect_id": aspect_id, "verdict": True, "raw_output": "ok",
        "latency_s": 0.01, "input_tokens": 1, "output_tokens": 1, "total_tokens": 2,
        "gpu_peak_mb": 1.0, "model_key": stage_c_verifier_key,
    }

det_problem_id = 0
det_slots = sample_slots(E2, dataset=stage_c_dataset_key, problem_id=det_problem_id, rep=0)
det_specs = [
    AgentSpec(agent_id=aid, model_key=mk, model_id=MODELS[mk]["model_id"], mode="replay",
              store=downloaded_stores[(mk, stage_c_dataset_key)], gen_id=gid)
    for aid, (mk, gid) in zip(e2_agent_ids, det_slots)
]
det_graph1 = build_bon_mav_graph(det_specs, stage_c_aspect_ids, _scripted_verify_for_determinism_check,
                                  select_fn, rng=random.Random(TIE_BREAK_SEED))
det_graph2 = build_bon_mav_graph(det_specs, stage_c_aspect_ids, _scripted_verify_for_determinism_check,
                                  select_fn, rng=random.Random(TIE_BREAK_SEED))
det_state = {
    "task": stage_c_spec.render_task(stage_c_sample[det_problem_id]), "problem_id": det_problem_id,
    "dataset": stage_c_dataset_key, "samples": [], "approvals": [], "selection": None,
}
det_out1 = det_graph1.invoke(dict(det_state))
det_out2 = det_graph2.invoke(dict(det_state))
checks.append(("BoN-MAV selection deterministic on re-invoke from a fresh seed",
                det_out1["selection"] == det_out2["selection"], ""))

# (n) verifier input_tokens context-length sanity — informational, not asserted: the
# 2-problem MATH smoke subset's short tasks/candidates make a lower-than-~3k number
# expected here, not a failure (full scale reprocesses a full ~2048-token candidate).
print(f"  Verifier input_tokens (BoN-MAV smoke): mean={mean_input_tokens:.0f} "
      f"(context-length sanity — expect ~3k at full scale, not at this tiny smoke scale)\n")

# (o) BoN-MAV's compose_generation_metrics VRAM composition adds exactly the
# non-verifier candidate models' peaks (plus the verifier's own) — every distinct
# model_key across the candidate half and the verify half, summed once each.
all_bonmav_calls = all_candidate_samples + all_approvals
bonmav_composed = compose_generation_metrics(all_bonmav_calls, tokens_per_s_rule="mean")
expected_model_keys = set(E2.slots) | {stage_c_verifier_key}
peak_by_model = {}
for call in all_bonmav_calls:
    peak_by_model[call["model_key"]] = max(peak_by_model.get(call["model_key"], 0.0), call["gpu_peak_mb"])
expected_vram = sum(peak_by_model.values())
checks.append((
    "BoN-MAV compose_generation_metrics VRAM == verifier peak + each distinct "
    "non-verifier candidate model's peak",
    abs(bonmav_composed["gpu_peak_mb"] - expected_vram) < 1e-6 and set(peak_by_model) == expected_model_keys,
    f"composed={bonmav_composed['gpu_peak_mb']:.1f} vs manual={expected_vram:.1f}, models={sorted(peak_by_model)}",
))

# (p) reused_gen_*/live_verify_* cost split (§2.5) was computed and logged in Stage C.
checks.append(("reused_gen_*/live_verify_* cost split computed and logged",
                "latency_s" in reused_gen_metrics and "latency_s" in live_verify_metrics, ""))

# (q) Stage C2 (BoN-MAV without SC-reuse, live candidates) ran end-to-end: the
# live-candidate approval matrix shape matches the same n_candidates x n_aspects x
# n_problems contract (k) already validates for the replayed case.
n_candidates_c2_expected = len(e2_agent_ids)
n_aspects_c2_expected = len(stage_c2_aspect_ids)
expected_c2_approval_count = n_candidates_c2_expected * n_aspects_c2_expected * len(stage_c2_sample)
checks.append((
    f"Stage C2 (live BoN-MAV) approval matrix shape == n_candidates x n_aspects per problem "
    f"({n_candidates_c2_expected}x{n_aspects_c2_expected}, {len(stage_c2_sample)} problems)",
    len(all_c2_approvals) == expected_c2_approval_count,
    f"got {len(all_c2_approvals)}, expected {expected_c2_approval_count}",
))

all_passed = True
for name, passed, detail in checks:
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"  {status}  {name}" + (f"   ({detail})" if detail else ""))
    all_passed = all_passed and passed

assert all_passed, "One or more invariant checks FAILED — see the table above."

print(f"\n{'='*70}\n  ALL INVARIANT CHECKS PASSED ✅\n{'='*70}")

## 15. Smoke Test Complete

If every cell above ran without an assertion failure, this run has validated:

- Full-text generation persistence and W&B artifact retrieval (`utils.persistence`) with real producer→consumer lineage.
- The LangGraph majority-voting template (`utils.graphs`) in live mode, for both MATH-500 and MMLU, including the Qwen3 `/no_think` path — the vote itself produced by the graph's `aggregate` node, via the shared `BenchmarkSpec` (`utils.voting.MATH500_SPEC`/`MMLU_SPEC`).
- The SC@k ladder (`utils.voting.run_self_consistency`, `compose_generation_metrics`) and its statistics/summary logging, matching 2.0.0 conventions.
- The primary key `(dataset, problem_id, model_key, gen_id)` is genuinely unique once every model x benchmark's records are pooled together — the on-Kaggle proof (`utils.persistence.assert_unique_pk`) that the readable-ID scheme holds under real data, not just synthetic fixtures.
- Sequential single-model VRAM discipline across 4 models.
- Deterministic, reproducible voting: two independent runs from a freshly seeded `random.Random(TIE_BREAK_SEED)` agree bit-for-bit.
- **Stage A** — Majority Voting (`utils.ensembles.sample_slots`, `build_majority_voting_graph`) replayed entirely from *downloaded* artifacts, with sampler/seed-reuse/replay-fidelity/determinism/composition invariants all holding, plus one full MAJ-style W&B run (results table, per-rep stats, npz artifact).
- **Stage B** — replaying SC generations is equivalent to running MAJ for real: `E1`'s 3 and `E4`'s 4 distinct models generated live and co-resident within the 16 GB budget, with the VRAM reconstruction rule validated against the raw co-resident peak.
- **Stage C** — the BoN-MAV template (`build_bon_mav_graph`) live end-to-end with a real verifier, exercising all 3 `max_new_tokens` strategy tiers, and producing the first real GPU-budget calibration data point.
- **Stage C2** — BoN-MAV live end-to-end with fully live (non-replayed) candidates from `E2`, quantifying the full-live cost against Stage C's `reused_gen_*`/`live_verify_*` split — the concrete evidence for how much SC-reuse saves.
- **Stage D** — the additional MAJ/BoN-MAV invariants (approval matrix shape, non-approval scoring of unparseable verdicts, selection determinism, VRAM composition, the `reused_gen_*`/`live_verify_*` cost split) folded into the checklist above.

**Next step:** record this notebook's W&B run URLs (generation runs, SC@k runs, the round-trip validation run, the Stage A/C/C2 runs — all tagged `smoke`) for the thesis's smoke-test evidence, then proceed to the 8 real SC notebooks (`v3.1.0`–`v3.3.1`), followed by `v4.1.0` (Majority Voting) and the `v5.x` BoN-MAV notebooks — the latter preceded by extending Stage C's calibration to ~10–20 real problems before scheduling any real session.